In [185]:
import pandas as pd
import config as cfg
import numpy as np

# Passengers Dataset

## 1) Importing data and skimming through

In [186]:
df_passengers = pd.read_csv(cfg.PASSENGERS_DATA)
df_passengers.head()

,NAZIONALITA,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,GIORNO_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,...,COMPAGNIA_AEREA,NUMERO_VOLO,ESITO_CONTROLLO,note_operatore,codice_rischio,Tipo Documento,FASCIA ETA,3nazionalita,compagnia%aerea,num volo
0,ALB,NAP,DUR,2024,02,13,2024-02-13 07:30:00,Napoli Capodichino,King Shaka International,Napoli,...,Fly Dubai,FZ1681,RESPINTO,NaN,NaN,Passaporto,N.D.,ALB,Fly Dubai,FZ1681
1,NaN,FCO,JFK,2024,01,22,2024-01-22 16:35:00,Fiumicino,John F Kennedy International,Roma,...,ITA Airways,AZ0609,NaN,NaN,NaN,Carta d'identità,18-30,ALB,ITA Airways,AZ0609
2,ALB,TSF,TIA,2024,02,4,2024-02-04 20:10:00,Treviso-Sant'Angelo,Rinas Mother Teresa,Treviso,...,Ryanair DAC,FR8400,SEGNALATO,NaN,NaN,N.D.,31-45,ALB,Ryanair DAC,FR8400
3,AFG,FCO,IST,2024,01,25,2024-01-25 13:05:00,Fiumicino,Havalimani,Roma,...,Turkish Airlines,TK1865,NaN,NaN,NaN,N.D.,61+,AFG,Turkish Airlines,TK1865
4,ALB,BGY,MLE,2024,02,13,FEB 13 2024,Orio al Serio,Male International,Bergamo,...,Fly Dubai,FZ1571,SEGNALATO,NaN,NaN,Permesso di soggiorno,46-60,ALB,Fly Dubai,FZ1571


In [187]:
display(df_passengers.columns, df_passengers.shape)
df_passengers.isnull().sum().sort_values(ascending=False)

Index(['NAZIONALITA', 'AREOPORTO_ARRIVO', 'AREOPORTO_PARTENZA',
       'ANNO_PARTENZA', 'MESE_PARTENZA', 'GIORNO_PARTENZA', 'DATA_PARTENZA',
       'DESCR_AEREOPORTO_ARR', 'DESCR_AEREOPORTO_PART', 'CITTA_ARR',
       'CITTA_PARTENZA', 'CODICE_PAESE_ARR', 'CODICE_PAESE_PART', 'PAESE_ARR',
       'PAESE_PART', 'ZONA', 'ENTRATI', 'INVESTIGATI', 'ALLARMATI',
       'TIPO_DOCUMENTO', 'GENERE', 'FASCIA_ETA', 'FLAG_TRANSITO',
       'COMPAGNIA_AEREA', 'NUMERO_VOLO', 'ESITO_CONTROLLO', 'note_operatore',
       'codice_rischio', 'Tipo Documento', 'FASCIA ETA', '3nazionalita',
       'compagnia%aerea', 'num volo'],
      dtype='object')

(5095, 33)

codice_rischio           5054
note_operatore           5034
ESITO_CONTROLLO          1289
NAZIONALITA               116
COMPAGNIA_AEREA            87
NUMERO_VOLO                70
TIPO_DOCUMENTO             62
GENERE                     45
INVESTIGATI                 0
compagnia%aerea             0
3nazionalita                0
FASCIA ETA                  0
Tipo Documento              0
FLAG_TRANSITO               0
FASCIA_ETA                  0
ALLARMATI                   0
ENTRATI                     0
AREOPORTO_ARRIVO            0
ZONA                        0
PAESE_PART                  0
PAESE_ARR                   0
CODICE_PAESE_PART           0
CODICE_PAESE_ARR            0
CITTA_PARTENZA              0
CITTA_ARR                   0
DESCR_AEREOPORTO_PART       0
DESCR_AEREOPORTO_ARR        0
DATA_PARTENZA               0
GIORNO_PARTENZA             0
MESE_PARTENZA               0
ANNO_PARTENZA               0
AREOPORTO_PARTENZA          0
num volo                    0
dtype: int

There are five duplicate columns that must be removed since they include inconsistent data. <br>
Then we can proceed with translating our columns.

In [188]:
to_be_dropped = ['FASCIA_ETA', 'TIPO_DOCUMENTO', 'NAZIONALITA', 'NUMERO_VOLO', 'COMPAGNIA_AEREA', 'ANNO_PARTENZA', 'MESE_PARTENZA', 'GIORNO_PARTENZA', 'note_operatore', 'codice_rischio']
df_passengers.drop(columns=to_be_dropped, errors='ignore', inplace=True)
df_passengers['GENERE'] = df_passengers['GENERE'].map(cfg.GENDER_MAPPING).fillna('Unknown')

In [189]:
df_passengers['ENTRATI'] = df_passengers['ENTRATI'].astype(str).str.replace('pax', '', case=False)
df_passengers['INVESTIGATI'] = df_passengers['INVESTIGATI'].astype(str).str.replace('pax', '', case=False)
df_passengers['ALLARMATI'] = df_passengers['ALLARMATI'].astype(str).str.replace('pax', '', case=False)

df_passengers['ENTRATI'] = df_passengers['ENTRATI'].str.replace('~', '').str.replace(',', '.')
df_passengers['INVESTIGATI'] = df_passengers['INVESTIGATI'].str.replace('~', '').str.replace(',', '.')
df_passengers['ALLARMATI'] = df_passengers['ALLARMATI'].str.replace('~', '').str.replace(',', '.')

df_passengers['ENTRATI'] = pd.to_numeric(df_passengers['ENTRATI'], errors='coerce').astype('Int64')
df_passengers['INVESTIGATI'] = pd.to_numeric(df_passengers['INVESTIGATI'], errors='coerce').astype('Int64')
df_passengers['ALLARMATI'] = pd.to_numeric(df_passengers['ALLARMATI'], errors='coerce').astype('Int64')

df_passengers = df_passengers[df_passengers['ENTRATI'] > 0]

# df_passengers['ENTRATI'] = df_passengers['ENTRATI'].apply(lambda x: x if x >= 0 else 0)
# df_passengers['INVESTIGATI'] = df_passengers['INVESTIGATI'].apply(lambda x: x if x >= 0 else 0)
# df_passengers['ALLARMATI'] = df_passengers['ALLARMATI'].apply(lambda x: x if x >= 0 else 0)

# df_passengers['ENTRATI'] = pd.to_numeric(df_passengers['ENTRATI'], errors='coerce').fillna(0).astype(int)
# df_passengers['INVESTIGATI'] = pd.to_numeric(df_passengers['INVESTIGATI'], errors='coerce').fillna(0).astype(int)
# df_passengers['ALLARMATI'] = pd.to_numeric(df_passengers['ALLARMATI'], errors='coerce').fillna(0).astype(int)

In [190]:
df_passengers['DATA_PARTENZA'] = df_passengers['DATA_PARTENZA'].str.upper().str.replace('GEN', 'JAN')
df_passengers['DATA_PARTENZA'] = pd.to_datetime(df_passengers['DATA_PARTENZA'], format='mixed', dayfirst=True)

In [191]:
df_passengers['CODICE_PAESE_ARR'] = df_passengers['CODICE_PAESE_ARR'].replace('IT', 'ITA')

In [192]:
placeholders = ['N.D.', 'n.d.', 'N/A', ' ', '?', '//']
df_passengers = df_passengers.replace(placeholders, np.nan)

In [193]:
df_passengers = df_passengers.drop_duplicates()

In [194]:
logic_filtering = (df_passengers['ALLARMATI'] <= df_passengers['INVESTIGATI']) & (df_passengers['INVESTIGATI'] <= df_passengers['ENTRATI'])
df_passengers = df_passengers[logic_filtering].copy()
df_passengers = df_passengers[df_passengers['ENTRATI'] <= 853]

In [195]:
df_passengers['AREOPORTO_ARRIVO'] = df_passengers['AREOPORTO_ARRIVO'].str.upper()
df_passengers['AREOPORTO_PARTENZA'] = df_passengers['AREOPORTO_PARTENZA'].str.upper()

In [196]:
df_passengers.rename(columns=cfg.COLUMN_MAPPING_PASSENGERS, inplace=True)
display(df_passengers.isnull().sum().sort_values(ascending=False))
df_passengers.shape

document_type                    1487
control_result                   1151
age_range                         466
arrival_airport_code                0
passengers_entries_count            0
airline                             0
nationality                         0
transit_flag                        0
gender                              0
passengers_flagged_count            0
passengers_investigated_count       0
zone                                0
departure_airport_code              0
departure_country                   0
arrival_country                     0
departure_country_code              0
arrival_country_code                0
departure_city                      0
arrival_city                        0
departure_airport_name              0
arrival_airport_name                0
departure_date                      0
flight_number                       0
dtype: int64

(4476, 23)

In [197]:
df_passengers.head()

,arrival_airport_code,departure_airport_code,departure_date,arrival_airport_name,departure_airport_name,arrival_city,departure_city,arrival_country_code,departure_country_code,arrival_country,...,passengers_investigated_count,passengers_flagged_count,gender,transit_flag,control_result,document_type,age_range,nationality,airline,flight_number
0,NAP,DUR,2024-02-13 07:30:00,Napoli Capodichino,King Shaka International,Napoli,Durban,ITA,ZAF,Italia,...,1,0,F,Singola Tratta,RESPINTO,Passaporto,NaN,ALB,Fly Dubai,FZ1681
2,TSF,TIA,2024-02-04 20:10:00,Treviso-Sant'Angelo,Rinas Mother Teresa,Treviso,Tirana,ITA,ALB,Italia,...,58,13,F,Singola Tratta,SEGNALATO,NaN,31-45,ALB,Ryanair DAC,FR8400
3,FCO,IST,2024-01-25 13:05:00,Fiumicino,Havalimani,Roma,Istanbul,ITA,TR,Italia,...,1,0,M,Singola Tratta,NaN,NaN,61+,AFG,Turkish Airlines,TK1865
4,BGY,MLE,2024-02-13 00:00:00,Orio al Serio,Male International,Bergamo,Male,ITA,MDV,Italia,...,2,1,F,Singola Tratta,SEGNALATO,Permesso di soggiorno,46-60,ALB,Fly Dubai,FZ1571
5,TSF,LTN,2024-02-18 16:30:00,Treviso-Sant'Angelo,London Luton,Treviso,Londra,ITA,GBR,Italia,...,3,0,F,Singola Tratta,RESPINTO,Carta d'identità,18-30,ALB,Ryanair DAC,FR1050


# Cases Dataset

In [198]:
df_cases = pd.read_csv(cfg.CASES_DATA)
display(df_cases.head())
display(df_cases.columns)
df_cases.shape

,OCCORRENZE,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,...,ZONA,TOT,MOTIVO_ALLARME,note_operatore,flag_rischio,Paese Partenza,CODICE PAESE ARR,3zona,paese%arr,tot voli
0,Voli con Allarmi,FCO,IST,2024,01,2024-01-30 09:15:00,Fiumicino,Havalimani,Roma,Istanbul,...,5,1,Manuale,NaN,NaN,Turchia,ITA,5,Italia,1
1,Viaggiatori con Allarmi,CIA,STN,2024,02,2024-02-03 13:15:00,Ciampino,Stansted,Roma,Londra,...,5,5,Manuale,NaN,NaN,Regno Unito,ITA,5,Italia,5
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024,01,2024-01-15 08:45:00,Fiumicino,London Heathrow,Roma,Londra,...,5,110,TSC,NaN,NaN,Regno Unito,ITA,5,Italia,110
3,Voli con Allarmi,MXP,LHR,2024,02,2024-02-02 08:40:00,Malpensa,London Heathrow,Milano,Londra,...,2,1,SDI,NaN,NaN,Regno Unito,ITA,2,Italia,1
4,Viaggiatori con Allarmi,PSA,BRS,2024,02,2024-02-16 12:50:00,Galileo Galilei,Bristol,Pisa,Bristol,...,8,2,INTERPOL,NaN,NaN,Regno Unito,ITA,8,Italia,2


Index(['OCCORRENZE', 'AREOPORTO_ARRIVO', 'AREOPORTO_PARTENZA', 'ANNO_PARTENZA',
       'MESE_PARTENZA', 'DATA_PARTENZA', 'DESCR_AEREOPORTO_ARR',
       'DESCR_AEREOPORTO_PART', 'CITTA_ARR', 'CITTA_PARTENZA',
       'CODICE_PAESE_ARR', 'CODICE_PAESE_PART', 'PAESE_ARR', 'PAESE_PART',
       'ZONA', 'TOT', 'MOTIVO_ALLARME', 'note_operatore', 'flag_rischio',
       'Paese Partenza', 'CODICE PAESE ARR', '3zona', 'paese%arr', 'tot voli'],
      dtype='object')

(5080, 24)

In [199]:
columns_to_be_dropped = ['PAESE_ARR', 'PAESE_PART', 'ZONA',
                         'note_operatore', 'flag_rischio','ANNO_PARTENZA', 
                         'MESE_PARTENZA', 'TOT', 'CODICE_PAESE_ARR']

df_cases.drop(columns=columns_to_be_dropped, errors='ignore', inplace=True)
display(df_cases.columns)
df_cases.shape

Index(['OCCORRENZE', 'AREOPORTO_ARRIVO', 'AREOPORTO_PARTENZA', 'DATA_PARTENZA',
       'DESCR_AEREOPORTO_ARR', 'DESCR_AEREOPORTO_PART', 'CITTA_ARR',
       'CITTA_PARTENZA', 'CODICE_PAESE_PART', 'MOTIVO_ALLARME',
       'Paese Partenza', 'CODICE PAESE ARR', '3zona', 'paese%arr', 'tot voli'],
      dtype='object')

(5080, 15)

In [200]:
df_cases['tot voli'] = df_cases['tot voli'].astype(str).str.extract(r'(\d+)').astype(float).fillna(0).astype(int)

In [201]:
df_cases['DATA_PARTENZA'] = df_cases['DATA_PARTENZA'].astype(str).str.upper().str.replace('GEN', 'JAN')
df_cases['DATA_PARTENZA'] = pd.to_datetime(df_cases['DATA_PARTENZA'], format='mixed', dayfirst=True, errors='coerce')

In [202]:
df_cases = df_cases.drop_duplicates()

In [203]:
df_cases['AREOPORTO_ARRIVO'] = df_cases['AREOPORTO_ARRIVO'].str.upper()
df_cases['AREOPORTO_PARTENZA'] = df_cases['AREOPORTO_PARTENZA'].str.upper()
df_cases['DESCR_AEREOPORTO_ARR'] = df_cases['DESCR_AEREOPORTO_ARR'].str.upper()
df_cases['DESCR_AEREOPORTO_PART'] = df_cases['DESCR_AEREOPORTO_PART'].str.upper()

In [204]:
placeholders = ['???', 'N/C', 'Altro']
df_cases['OCCORRENZE'] = df_cases['OCCORRENZE'].replace(placeholders, 'UNKNOWN')

In [205]:
# df_cases[df_cases['DESCR_AEREOPORTO_PART'].str.len() < 5]['DESCR_AEREOPORTO_PART'].unique()
# df_cases[df_cases['DESCR_AEREOPORTO_ARR'].str.len() < 5]['DESCR_AEREOPORTO_ARR'].unique()


In [206]:
placeholders = [' ', '?', 'N.D.', 'ND', '-', '//', 'N/A ', '  ', '- ', 'N/A', '','NULL']
df_cases['DESCR_AEREOPORTO_PART'] = df_cases['DESCR_AEREOPORTO_PART'].str.upper().str.strip()
df_cases['DESCR_AEREOPORTO_PART'] = df_cases['DESCR_AEREOPORTO_PART'].replace(placeholders, np.nan)
df_cases['DESCR_AEREOPORTO_PART'] = df_cases['DESCR_AEREOPORTO_PART'].fillna(df_cases['AREOPORTO_PARTENZA'])

In [207]:
placeholders = ['n.d.', 'ND', '?', 'unknown', 'EU', 'XX', '00', '//', '-', 'ZZ', ' ']
df_cases['CODICE_PAESE_PART'] = df_cases['CODICE_PAESE_PART'].replace(cfg.COUNTRY_CODES)
df_cases['CODICE_PAESE_PART'] = df_cases['CODICE_PAESE_PART'].replace(placeholders, 'UNKNOWN')
df_cases['CODICE_PAESE_PART'].fillna('UNKNOWN', inplace=True)
df_cases['CODICE_PAESE_PART'] = df_cases['CODICE_PAESE_PART'].str.upper()

reference_map = df_cases[df_cases['CODICE_PAESE_PART'] != 'UNKNOWN'].set_index('Paese Partenza')['CODICE_PAESE_PART'].to_dict()
df_cases.loc[df_cases['CODICE_PAESE_PART'] == 'UNKNOWN', 'CODICE_PAESE_PART'] = df_cases['Paese Partenza'].map(reference_map)

df_cases['CITTA_PARTENZA'].fillna('unknown', inplace=True)
df_cases['CITTA_PARTENZA'] = df_cases.apply(lambda row: f"UNKNOWN ({row['Paese Partenza']})" if row['CITTA_PARTENZA'] in placeholders else row['CITTA_PARTENZA'],axis=1)

C:\Users\AFGWO\AppData\Local\Temp\ipykernel_14612\1345999424.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cases['CODICE_PAESE_PART'].fillna('UNKNOWN', inplace=True)
C:\Users\AFGWO\AppData\Local\Temp\ipykernel_14612\1345999424.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a 

In [208]:
df_cases['Paese Partenza'] = df_cases['Paese Partenza'].str.upper()
df_cases['CITTA_ARR'] = df_cases['CITTA_ARR'].str.upper()
df_cases['CITTA_PARTENZA'] = df_cases['CITTA_PARTENZA'].str.upper()
df_cases['paese%arr'] = df_cases['paese%arr'].str.upper()
df_cases['MOTIVO_ALLARME'] = df_cases['MOTIVO_ALLARME'].str.upper()

In [209]:
country_zone_map = df_cases.groupby('CODICE_PAESE_PART')['3zona'].agg(lambda x: x.mode().iloc[0]).to_dict()
df_cases['3zona'] = df_cases['CODICE_PAESE_PART'].map(country_zone_map)

In [210]:
df_cases.rename(columns=cfg.COLUMN_MAPPING_CASES, inplace=True)
display(df_cases.isnull().sum().sort_values(ascending=False))
df_cases.shape

alarm_reason              1143
event_type                   0
arrival_airport_code         0
departure_airport_code       0
departure_date               0
arrival_airport_name         0
departure_airport_name       0
arrival_city_name            0
departure_city_name          0
departure_country_code       0
departure_country_name       0
arrival_country_code         0
region_zone                  0
arrival_country_name         0
total_flights                0
dtype: int64

(5030, 15)

In [211]:
df_cases.head()

,event_type,arrival_airport_code,departure_airport_code,departure_date,arrival_airport_name,departure_airport_name,arrival_city_name,departure_city_name,departure_country_code,alarm_reason,departure_country_name,arrival_country_code,region_zone,arrival_country_name,total_flights
0,Voli con Allarmi,FCO,IST,2024-01-30 09:15:00,FIUMICINO,HAVALIMANI,ROMA,ISTANBUL,TUR,MANUALE,TURCHIA,ITA,5,ITALIA,1
1,Viaggiatori con Allarmi,CIA,STN,2024-02-03 13:15:00,CIAMPINO,STANSTED,ROMA,LONDRA,GBR,MANUALE,REGNO UNITO,ITA,2,ITALIA,5
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024-01-15 08:45:00,FIUMICINO,LONDON HEATHROW,ROMA,LONDRA,GBR,TSC,REGNO UNITO,ITA,2,ITALIA,110
3,Voli con Allarmi,MXP,LHR,2024-02-02 08:40:00,MALPENSA,LONDON HEATHROW,MILANO,LONDRA,GBR,SDI,REGNO UNITO,ITA,2,ITALIA,1
4,Viaggiatori con Allarmi,PSA,BRS,2024-02-16 12:50:00,GALILEO GALILEI,BRISTOL,PISA,BRISTOL,GBR,INTERPOL,REGNO UNITO,ITA,2,ITALIA,2


In [212]:
df_cases.to_csv(cfg.CASES_CLEAN_OUT)
df_passengers.to_csv(cfg.PASSENGERS_CLEAN_OUT)

# Table Formation

In [213]:
df_passengers.columns

Index(['arrival_airport_code', 'departure_airport_code', 'departure_date',
       'arrival_airport_name', 'departure_airport_name', 'arrival_city',
       'departure_city', 'arrival_country_code', 'departure_country_code',
       'arrival_country', 'departure_country', 'zone',
       'passengers_entries_count', 'passengers_investigated_count',
       'passengers_flagged_count', 'gender', 'transit_flag', 'control_result',
       'document_type', 'age_range', 'nationality', 'airline',
       'flight_number'],
      dtype='object')

In [214]:
df_passengers["departure_day"] = df_passengers["departure_date"].dt.floor("D")


In [215]:
base_keys = [
    "departure_day",
    "arrival_airport_code",
    "departure_airport_code",
    "nationality",
    "document_type"
]

base_passenger = (
    df_passengers
    .groupby(base_keys, dropna=False, as_index=False)
    .agg(
        entries_count=("passengers_entries_count", "sum"),
        investigated_count=("passengers_investigated_count", "sum"),
        flagged_count=("passengers_flagged_count", "sum")
    )
)

display(base_passenger.head())
display(base_passenger.shape)

,departure_day,arrival_airport_code,departure_airport_code,nationality,document_type,entries_count,investigated_count,flagged_count
0,2023-12-31,FCO,JFK,ALB,Carta d'identità,3,3,0
1,2024-01-01,AOI,TIA,ALB,Permesso di soggiorno,50,50,8
2,2024-01-01,BDS,TIA,ALB,Visto,34,34,2
3,2024-01-01,BGY,BEG,ALB,Visto,1,1,0
4,2024-01-01,BGY,BFS,ALB,Permesso di soggiorno,1,1,0


(4100, 8)

In [216]:
base_passenger[~base_passenger.isnull()]

base_passenger["investigation_rate"] = base_passenger["investigated_count"] / base_passenger["entries_count"]
base_passenger["flag_rate"] = base_passenger["flagged_count"] / base_passenger["entries_count"]
base_passenger["flag_given_investigation_rate"] = base_passenger["flagged_count"] / base_passenger["investigated_count"]

base_passenger[base_passenger['entries_count'] >= 25].sort_values(by=['flag_rate','flag_given_investigation_rate'],ascending=False).head(20)


,departure_day,arrival_airport_code,departure_airport_code,nationality,document_type,entries_count,investigated_count,flagged_count,investigation_rate,flag_rate,flag_given_investigation_rate
3849,2024-02-25,TSF,TIA,ALB,Visto,30,30,13,1.0,0.433333,0.433333
145,2024-01-02,TSF,TIA,ALB,Carta d'identità,39,39,15,1.0,0.384615,0.384615
2842,2024-02-11,BLQ,TIA,ALB,Permesso di soggiorno,39,39,14,1.0,0.358974,0.358974
3863,2024-02-26,BGY,TIA,ALB,Carta d'identità,49,49,17,1.0,0.346939,0.346939
3362,2024-02-18,PSA,TIA,ALB,Passaporto,123,123,41,1.0,0.333333,0.333333
3834,2024-02-25,MXP,TIA,ALB,Passaporto,81,81,26,1.0,0.320988,0.320988
2426,2024-02-04,MXP,TIA,ALB,Carta d'identità,53,53,17,1.0,0.320755,0.320755
272,2024-01-04,FLR,TIA,ALB,Visto,78,78,25,1.0,0.320513,0.320513
1698,2024-01-24,TSF,TIA,ALB,Visto,44,44,14,1.0,0.318182,0.318182
1513,2024-01-21,MXP,TIA,ALB,Permesso di soggiorno,76,76,24,1.0,0.315789,0.315789


In [217]:
route_day = (
    base_passenger
    .groupby(
        ["departure_day", "arrival_airport_code", "departure_airport_code"],
        dropna=False,
        as_index=False
    )
    .agg(
        route_entries_count=("entries_count", "sum"),
        route_investigated_count=("investigated_count", "sum"),
        route_flagged_count=("flagged_count", "sum")
    )
)

route_day["route_investigation_rate"] = route_day["route_investigated_count"] / route_day["route_entries_count"]
route_day["route_flag_rate"] = route_day["route_flagged_count"] / route_day["route_entries_count"]

route_day[route_day['route_entries_count'] >= 25].sort_values(by=['route_investigation_rate', 'route_flag_rate'], ascending=False).head(20)

,departure_day,arrival_airport_code,departure_airport_code,route_entries_count,route_investigated_count,route_flagged_count,route_investigation_rate,route_flag_rate
2992,2024-02-25,TSF,TIA,30,30,13,1.0,0.433333
204,2024-01-04,FLR,TIA,78,78,25,1.0,0.320513
2232,2024-02-11,VRN,TIA,86,86,27,1.0,0.313953
2025,2024-02-08,AOI,TIA,64,64,20,1.0,0.3125
1625,2024-01-30,NAP,TIA,68,68,20,1.0,0.294118
2773,2024-02-22,CIA,TIA,65,65,19,1.0,0.292308
294,2024-01-06,AOI,TIA,103,103,30,1.0,0.291262
118,2024-01-02,VCE,TIA,59,59,17,1.0,0.288136
266,2024-01-05,GOA,TIA,116,116,33,1.0,0.284483
2661,2024-02-20,AOI,TIA,89,89,25,1.0,0.280899


In [218]:
nationality_day = (
    base_passenger
    .groupby(["departure_day", "nationality"], dropna=False, as_index=False)
    .agg(
        nationality_entries_count=("entries_count", "sum"),
        nationality_investigated_count=("investigated_count", "sum"),
        nationality_flagged_count=("flagged_count", "sum")
    )
)

nationality_day["nationality_investigation_rate"] = nationality_day["nationality_investigated_count"] / nationality_day["nationality_entries_count"]
nationality_day["nationality_flag_rate"] = nationality_day["nationality_flagged_count"] / nationality_day["nationality_entries_count"]

nationality_day.sort_values(by=['nationality_investigation_rate', 'nationality_flag_rate'], ascending=False).head(20)

,departure_day,nationality,nationality_entries_count,nationality_investigated_count,nationality_flagged_count,nationality_investigation_rate,nationality_flag_rate
81,2024-01-25,AND,1,1,1,1.0,1.0
98,2024-01-31,AFG,10,10,8,1.0,0.8
119,2024-02-05,AFG,11,11,7,1.0,0.636364
42,2024-01-13,AFG,10,10,6,1.0,0.6
148,2024-02-14,AFG,12,12,7,1.0,0.583333
161,2024-02-18,AFG,11,11,6,1.0,0.545455
198,2024-02-29,AFG,13,13,7,1.0,0.538462
68,2024-01-22,AFG,4,4,2,1.0,0.5
95,2024-01-30,AFG,8,8,4,1.0,0.5
123,2024-02-06,AFG,4,4,2,1.0,0.5


In [219]:
nationality = (
    base_passenger
    .groupby(["nationality"], dropna=False, as_index=False)
    .agg(
        nationality_entries_count=("entries_count", "sum"),
        nationality_investigated_count=("investigated_count", "sum"),
        nationality_flagged_count=("flagged_count", "sum")
    )
)

nationality["nationality_investigation_rate"] = nationality["nationality_investigated_count"] / nationality["nationality_entries_count"]
nationality["nationality_flag_rate"] = nationality["nationality_flagged_count"] / nationality["nationality_entries_count"]

nationality.sort_values(by=['nationality_investigation_rate', 'nationality_flag_rate'], ascending=False).head(20)

,nationality,nationality_entries_count,nationality_investigated_count,nationality_flagged_count,nationality_investigation_rate,nationality_flag_rate
5,AND,38,38,1,1.0,0.026316
3,AIA,1,1,0,1.0,0.0
6,ARE,54,54,0,1.0,0.0
1,AFG,878,859,207,0.97836,0.235763
4,ALB,160944,156047,22500,0.969573,0.1398
2,AGO,60,51,4,0.85,0.066667
0,ABW,2,1,0,0.5,0.0
